This is one big problem for self-assessment that checks your understanding for the following topics:
- Python Syntax and Data Types
- Python Standard Library:
    - `itertools`
    - `collections`
    - `datetime`: date representations and operations
    - `pathlib`: file and directory operations
    - `json` (including reading from url)
    - and perhaps some others
- File I/O
- Data Types and Algorithms
- A bit of visualization with `matplotlib`


Prerequisites:
- builtins: parts 1-5
- comprehensions

Short description of the tasks by difficulty:
- Basic:
    - read all data from the local directory and urls
    - personal user statistics: age, height, weight, education, occupation, etc.
- Intermediate:
    - some kind of 1D linear regression analysis
    - group and match users with their posts, comments and todos
- Advanced:
    - read the icons from the urls provided in the json files
    - plot the user points on the map from the coordinates provided in the json files
- Expert:
    - build the network graph of the users and their connections; then return the friends of friends of a given user
    - giveaways: a user gives each follower some amount of money; calculate the resulting balance of each user

# Social Network Data

We aim to simulate a small social network with about 200 users: their personal information, posts, comments, todos, connections and other data.

Follow the instructions and analyse the data located in different sources and formats.

In [ ]:
# note: you may or may not need all of these
import pathlib
import itertools
import collections
import datetime
import json
import csv
import urllib.request

import matplotlib.pyplot as plt

from utils.currency import convert_currency

## Main Network Data (url)

Here are the url links to the json data files containing (aftificial) information about a social network's users, their posts, comments, reactions and private TODO lists.

In [ ]:
USERS_URL = "https://raw.githubusercontent.com/Ovi/DummyJSON/master/database/users.json"
POSTS_URL = "https://raw.githubusercontent.com/Ovi/DummyJSON/master/database/posts.json"
COMMENTS_URL = (
    "https://raw.githubusercontent.com/Ovi/DummyJSON/master/database/comments.json"
)
TODOS_URL = "https://raw.githubusercontent.com/Ovi/DummyJSON/master/database/todos.json"

#* note: you should not download the data, but read it directly from the web

## Local Data

There is some more data stored locally in the `data` directory.

In [ ]:
DATA_DIR = pathlib.Path("data")

for file in DATA_DIR.iterdir():
    print(file)

There the file `network.csv` contains the information about the (asymmetric) connections between the users.

```csv
follower,followed
30,21
21,50
...
```

In the example above, user 30 follows user 21, and user 21 follows user 50.

The file `bank_data.csv` contains the information about the bank accounts of the users.

```csv
iban,balance
1GBYGX4DZ5G4ZXXKNTELQ6TL,39284.35
...
```

Note that the account balance is given in that account's currency.

The files of format `transactions_<number>.jsonl` contain the information about the transaction requests between the users.

```jsonl
{"timestamp": 1448576423, "sender": "1GBYGX4DZ5G4ZXXKNTELQ6TL", "receiver": "RUZOEQNPEYQ110WTGXTBMUA6", "amount": 23.99}
{"timestamp": ..., ...}
```

Things to note:
- the transactions are always in the currency of the sender's account (e.g. the first transaction above is from an account with balance in USD to an account with balance in BRL);
- the transaction amount is always positive;
- the transaction amount is always rounded to 2 decimal places;

moreover, some transaction requests are not valid and thus **cannot** be completed due to
- insufficient balance on the sender's account (error code `1`)
- the receiver's account does not exist (error code `2`)
- the sender's or the receiver's account is expired when the transaction is issued ("cardExpire" date is in the past) (error code `3`)

Since quite a bit of data has accumulated over the years, we won't provide the raw files. Run the following cell to generate the pseudo-random data for this project.

_Note_: this will take a while (there are 10 million transactions).

In [ ]:
from utils._gen import generate_transactions


DEST = DATA_DIR / "transactions"
generate_transactions(DEST)

Take a look inside at the transactions data (do not share: it's highly confidential).

# Tasks

## Task 0. Complete the utility functions

In the `utils` module, there are some utility functions that you need to complete.

After you do so, restart the kernel and test them in the cell below.

### 0.1. `currency.convert_currency`

In [ ]:
# test
assert convert_currency(1.00, "USD", "USD") == 1.00
assert convert_currency(45.23, "USD", "EUR") == 41.57
assert convert_currency(107.5, "GBP", "CAD") == 190.6
assert convert_currency(9.95, "TRY", "INR") == 25.12
assert convert_currency(5.00, "JPY", "JPY") == 5.00


## Task 1. Reading the Data

### Subtask 1.1. Read the Data from the URLs

In [ ]:
USERS_DATA: list[dict] = json.loads(urllib.request.urlopen(USERS_URL).read())
POSTS_DATA: list[dict] = json.loads(urllib.request.urlopen(POSTS_URL).read())
COMMENTS_DATA: list[dict] = json.loads(urllib.request.urlopen(COMMENTS_URL).read())
TODOS_DATA: list[dict] = json.loads(urllib.request.urlopen(TODOS_URL).read())

In [ ]:
# test user data
assert len(USERS_DATA) == 208
assert USERS_DATA[0]["id"] == 1
assert USERS_DATA[0]["firstName"] == "Emily"
assert USERS_DATA[0]["lastName"] == "Johnson"
assert sum(u['weight'] for u in USERS_DATA) == 15608.03

In [ ]:
# test post data
assert len(POSTS_DATA) == 251

In [ ]:
# test comment data


In [ ]:
# test todo data


### Subtask 1.2. Read the Data from the Local Files

In [ ]:
def load_bank_data() -> dict[str, float]:
    """Load the bank data from the bank_data.csv file.
    The resulting object maps IBANs to balances in the account's currency.
    """
    FILE = DATA_DIR / "bank_data.csv"
    with open(FILE) as file:
        reader = csv.DictReader(file)
        return {row["iban"]: float(row["balance"]) for row in reader}


In [ ]:
BANK_DATA = load_bank_data()

In [ ]:
import math

assert len(BANK_DATA) == 208
assert math.isclose(sum(BANK_DATA.values()), 62369452.29)
assert BANK_DATA["7N7ZH1PJ8Q4WU1K965HQQR27"] == 33492.23


## Task 2. Personal Data Analysis
- average age, height and weight
- most common occupation and education level
- blood type distribution

## Task 3. Posts Analysis
- overall likes ratio distribution (histogram)
- all unique tags and their counts
- tag influence on likes ratio